# Multi-Layer Perceptron

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')


## Load Data

In [3]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
valid_df = pd.read_csv(os.path.join(data_dir, 'valid_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]


### Danh sách lưu kết quả để xuất CSV

In [4]:
results_list = []

def evaluate_model(model, name, X_split, y_split, split_name):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)."""
    y_pred = model.predict(X_split)
    y_prob = model.predict_proba(X_split)[:, 1]  # Xác suất class 1 để tính AUC

    acc = accuracy_score(y_split, y_pred)
    prec = precision_score(y_split, y_pred)
    rec = recall_score(y_split, y_pred)
    f1 = f1_score(y_split, y_pred)
    auc = roc_auc_score(y_split, y_prob)

    res = {
        'Algorithm': name,
        'Split': split_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'AUC': round(auc, 4)
    }

    results_list.append(res)
    print(f"[{name} | {split_name}] Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    return res


In [5]:
# K-FOLD CONFIG
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_kfold(model, name, X_data, y_data, cv_split):
    """Đánh giá K-Fold và lưu kết quả trung bình vào results_list."""
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'auc': 'roc_auc'
    }

    cv_results = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv_split,
        scoring=scoring,
        n_jobs=-1
    )

    res = {
        'Algorithm': name,
        'Split': 'KFold',
        'Accuracy': round(cv_results['test_accuracy'].mean(), 4),
        'Precision': round(cv_results['test_precision'].mean(), 4),
        'Recall': round(cv_results['test_recall'].mean(), 4),
        'F1-Score': round(cv_results['test_f1'].mean(), 4),
        'AUC': round(cv_results['test_auc'].mean(), 4)
    }

    results_list.append(res)
    print(
        f"[{name} | KFold] Acc: {res['Accuracy']:.4f} | Prec: {res['Precision']:.4f} | "
        f"Rec: {res['Recall']:.4f} | F1: {res['F1-Score']:.4f} | AUC: {res['AUC']:.4f}"
    )
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [6]:
print("--- V1: Baseline Multi-Layer Perceptron ---")
MLP_baseline_model = MLPClassifier(random_state=42, max_iter=500)

# Đánh giá K-Fold trước
evaluate_kfold(MLP_baseline_model, "V1: Multi-Layer Perceptron Baseline", X_train, y_train, kfold)

# Train cố định trên train set rồi đánh giá validation/test
MLP_baseline_model.fit(X_train, y_train)
evaluate_model(MLP_baseline_model, "V1: Multi-Layer Perceptron Baseline", X_valid, y_valid, "Validation")
evaluate_model(MLP_baseline_model, "V1: Multi-Layer Perceptron Baseline", X_test, y_test, "Test")


--- V1: Baseline Multi-Layer Perceptron ---
[V1: Multi-Layer Perceptron Baseline | KFold] Acc: 0.9959 | Prec: 0.9912 | Rec: 1.0000 | F1: 0.9955 | AUC: 1.0000
[V1: Multi-Layer Perceptron Baseline | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V1: Multi-Layer Perceptron Baseline | Test] Acc: 0.9877 | Prec: 0.9734 | Rec: 1.0000 | F1: 0.9865 | AUC: 1.0000


{'Algorithm': 'V1: Multi-Layer Perceptron Baseline',
 'Split': 'Test',
 'Accuracy': 0.9877,
 'Precision': 0.9734,
 'Recall': 1.0,
 'F1-Score': 0.9865,
 'AUC': 1.0}

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [7]:
print("--- V2: GridSearchCV Tuning ---")
param_grid = {'hidden_layer_sizes': [(50,), (100,), (50, 50)], 'alpha': [0.0001, 0.001, 0.01], 'learning_rate_init': [0.001, 0.01]}

grid_search = GridSearchCV(
    MLPClassifier(random_state=42, max_iter=500),
    param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
MLP_tuned_model = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")

# Đánh giá K-Fold cho mô hình tốt nhất
evaluate_kfold(MLP_tuned_model, "V2: Multi-Layer Perceptron Tuned (GridSearch)", X_train, y_train, kfold)

evaluate_model(MLP_tuned_model, "V2: Multi-Layer Perceptron Tuned (GridSearch)", X_valid, y_valid, "Validation")
evaluate_model(MLP_tuned_model, "V2: Multi-Layer Perceptron Tuned (GridSearch)", X_test, y_test, "Test")


--- V2: GridSearchCV Tuning ---
Best Params found by CV: {'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'learning_rate_init': 0.01}
[V2: Multi-Layer Perceptron Tuned (GridSearch) | KFold] Acc: 0.9986 | Prec: 0.9972 | Rec: 1.0000 | F1: 0.9986 | AUC: 1.0000
[V2: Multi-Layer Perceptron Tuned (GridSearch) | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V2: Multi-Layer Perceptron Tuned (GridSearch) | Test] Acc: 0.9975 | Prec: 0.9946 | Rec: 1.0000 | F1: 0.9973 | AUC: 1.0000


{'Algorithm': 'V2: Multi-Layer Perceptron Tuned (GridSearch)',
 'Split': 'Test',
 'Accuracy': 0.9975,
 'Precision': 0.9946,
 'Recall': 1.0,
 'F1-Score': 0.9973,
 'AUC': 1.0}

In [8]:
print("--- V3: Ensemble Learning (Stacking) ---")
# Kết hợp mô hình chính và KNN làm Base Models
base_estimators = [('mlp', MLPClassifier(random_state=42, max_iter=500)), ('knn', KNeighborsClassifier(n_neighbors=5))]

MLP_stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=kfold
)

# Đánh giá K-Fold cho stacking
evaluate_kfold(MLP_stacking_model, "V3: Multi-Layer Perceptron Stacking Ensemble", X_train, y_train, kfold)

MLP_stacking_model.fit(X_train, y_train)
evaluate_model(MLP_stacking_model, "V3: Multi-Layer Perceptron Stacking Ensemble", X_valid, y_valid, "Validation")
evaluate_model(MLP_stacking_model, "V3: Multi-Layer Perceptron Stacking Ensemble", X_test, y_test, "Test")


--- V3: Ensemble Learning (Stacking) ---
[V3: Multi-Layer Perceptron Stacking Ensemble | KFold] Acc: 0.9959 | Prec: 0.9912 | Rec: 1.0000 | F1: 0.9955 | AUC: 1.0000
[V3: Multi-Layer Perceptron Stacking Ensemble | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V3: Multi-Layer Perceptron Stacking Ensemble | Test] Acc: 0.9975 | Prec: 0.9946 | Rec: 1.0000 | F1: 0.9973 | AUC: 1.0000


{'Algorithm': 'V3: Multi-Layer Perceptron Stacking Ensemble',
 'Split': 'Test',
 'Accuracy': 0.9975,
 'Precision': 0.9946,
 'Recall': 1.0,
 'F1-Score': 0.9973,
 'AUC': 1.0}

### (5) Lưu kết quả

In [9]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/MLP/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU 1 FILE CSV DUY NHẤT
df_results = pd.DataFrame(results_list)

csv_output = os.path.join(save_path, 'MLP_full_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'MLP_baseline.pkl'), 'wb') as f:
    pickle.dump(MLP_baseline_model, f)

with open(os.path.join(save_path, 'MLP_tuned.pkl'), 'wb') as f:
    pickle.dump(MLP_tuned_model, f)

with open(os.path.join(save_path, 'MLP_stacking.pkl'), 'wb') as f:
    pickle.dump(MLP_stacking_model, f)

print("-" * 40)
print(f"✅ Đã lưu tất cả kết quả vào: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)


----------------------------------------
✅ Đã lưu tất cả kết quả vào: ../experiment/MLP/MLP_full_results.csv
✅ Đã lưu models tại: ../experiment/MLP/
----------------------------------------


,Algorithm,Split,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: Multi-Layer Perceptron Baseline,KFold,0.9959,0.9912,1.0,0.9955,1.0
1,V1: Multi-Layer Perceptron Baseline,Validation,1.0000,1.0000,1.0,1.0000,1.0
2,V1: Multi-Layer Perceptron Baseline,Test,0.9877,0.9734,1.0,0.9865,1.0
3,V2: Multi-Layer Perceptron Tuned (GridSearch),KFold,0.9986,0.9972,1.0,0.9986,1.0
4,V2: Multi-Layer Perceptron Tuned (GridSearch),Validation,1.0000,1.0000,1.0,1.0000,1.0
5,V2: Multi-Layer Perceptron Tuned (GridSearch),Test,0.9975,0.9946,1.0,0.9973,1.0
6,V3: Multi-Layer Perceptron Stacking Ensemble,KFold,0.9959,0.9912,1.0,0.9955,1.0
7,V3: Multi-Layer Perceptron Stacking Ensemble,Validation,1.0000,1.0000,1.0,1.0000,1.0
8,V3: Multi-Layer Perceptron Stacking Ensemble,Test,0.9975,0.9946,1.0,0.9973,1.0


In [10]:
!jupyter nbconvert --to html MLP_model.ipynb

[NbConvertApp] Converting notebook MLP_model.ipynb to html
[NbConvertApp] Writing 309052 bytes to MLP_model.html
